# 🏥 CNN-Based ICD-10 Auto-Coding System
## Notebook 4: Inference & Deployment

---

**This Notebook Covers**:
1. Load trained model from Google Drive
2. Extract text from new PDF documents
3. Generate ICD-10 code predictions
4. Interactive demo for testing

**Input**: New PDF documents or text
**Output**: Predicted ICD-10 codes with confidence scores

**Estimated Runtime**: ~5 min setup, then instant predictions

---

## 📋 Section 1: Setup & Load Model (~3 min)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✓ Drive mounted!")

In [ ]:
%%capture
# Install dependencies
!pip install pdfplumber pytesseract pdf2image nltk
!apt-get install -y tesseract-ocr poppler-utils
print("✓ Dependencies installed!")

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pickle
import json
import re
import os
from typing import List, Dict, Tuple

# PDF processing
import pdfplumber
from pdf2image import convert_from_path
import pytesseract

# NLP
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

DRIVE_BASE = "/content/drive/MyDrive"
PROJECT_FOLDER = f"{DRIVE_BASE}/ICD10_Project"
MODELS_FOLDER = f"{PROJECT_FOLDER}/models"
DATA_FOLDER = f"{PROJECT_FOLDER}/data/train_test_split"

# Model and preprocessor files
MODEL_PATH = f"{MODELS_FOLDER}/icd10_cnn_latest.pt"
VOCAB_PATH = f"{DATA_FOLDER}/vocabulary.pkl"
LABEL_ENCODER_PATH = f"{DATA_FOLDER}/label_encoder.pkl"

# Prediction settings
THRESHOLD = 0.3  # Lower threshold for better recall
TOP_K = 10       # Max predictions per document

print("Configuration set!")
print(f"  Model: {MODEL_PATH}")
print(f"  Threshold: {THRESHOLD}")

In [ ]:
# ============================================
# DEFINE CNN MODEL (same architecture as training)
# ============================================

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_classes,
                 num_filters, kernel_sizes, dropout_rate, hidden_dim):
        super(TextCNN, self).__init__()
        
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        
        total_filters = num_filters * len(kernel_sizes)
        self.fc1 = nn.Linear(total_filters, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_classes)
        
        self.dropout = nn.Dropout(dropout_rate)
        self.bn = nn.BatchNorm1d(hidden_dim)
        
    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)
        
        conv_outputs = []
        for conv in self.convs:
            c = F.relu(conv(x))
            c = F.max_pool1d(c, c.size(2)).squeeze(2)
            conv_outputs.append(c)
        
        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        x = F.relu(self.bn(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return torch.sigmoid(x)

print("✓ TextCNN class defined!")

In [ ]:
# Load model
print("Loading trained model...")

# Load saved model info
model_info = torch.load(MODEL_PATH, map_location=device)
config = model_info['model_config']

# Create model with saved configuration
model = TextCNN(
    vocab_size=config['vocab_size'],
    embedding_dim=config['embedding_dim'],
    n_classes=config['n_classes'],
    num_filters=config['num_filters'],
    kernel_sizes=config['kernel_sizes'],
    dropout_rate=config['dropout_rate'],
    hidden_dim=config['hidden_dim']
).to(device)

# Load trained weights
model.load_state_dict(model_info['model_state_dict'])
model.eval()

print(f"\n✓ Model loaded!")
print(f"  Classes: {config['n_classes']}")
print(f"  Best Val F1: {model_info['metrics']['best_val_f1']:.4f}")

In [ ]:
# Load vocabulary and label encoder
print("Loading vocabulary and label encoder...")

with open(VOCAB_PATH, 'rb') as f:
    vocab = pickle.load(f)

with open(LABEL_ENCODER_PATH, 'rb') as f:
    label_encoder = pickle.load(f)

idx2code = label_encoder['idx2code']
code2idx = label_encoder['code2idx']
MAX_SEQ_LENGTH = 2000

print(f"✓ Vocabulary size: {len(vocab)}")
print(f"✓ Number of codes: {len(idx2code)}")

---

## 🔧 Section 2: Preprocessing Pipeline (~1 min)

In [ ]:
class ICD10Predictor:
    """
    End-to-end predictor for ICD-10 codes.
    Handles text preprocessing and model inference.
    """
    
    def __init__(self, model, vocab, idx2code, device, max_seq_length=2000):
        self.model = model
        self.vocab = vocab
        self.idx2code = idx2code
        self.device = device
        self.max_seq_length = max_seq_length
        
        # NLP tools
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        
        # Medical terms to keep
        medical_keep = {'pain', 'left', 'right', 'upper', 'lower', 'chronic', 'acute',
                       'mild', 'moderate', 'severe', 'normal', 'abnormal', 'positive',
                       'negative', 'stable', 'unstable', 'primary', 'secondary'}
        self.stop_words -= medical_keep
        
        # ICD-10 chapter descriptions
        self.icd10_chapters = {
            'A': 'Infectious diseases', 'B': 'Infectious diseases',
            'C': 'Neoplasms', 'D': 'Blood diseases',
            'E': 'Endocrine/Metabolic', 'F': 'Mental disorders',
            'G': 'Nervous system', 'H': 'Eye/Ear',
            'I': 'Circulatory system', 'J': 'Respiratory system',
            'K': 'Digestive system', 'L': 'Skin diseases',
            'M': 'Musculoskeletal', 'N': 'Genitourinary',
            'O': 'Pregnancy', 'P': 'Perinatal conditions',
            'Q': 'Congenital malformations', 'R': 'Symptoms/Signs',
            'S': 'Injury', 'T': 'Injury/Poisoning',
            'V': 'External causes', 'W': 'External causes',
            'X': 'External causes', 'Y': 'External causes',
            'Z': 'Factors influencing health'
        }
    
    def preprocess_text(self, text: str) -> List[str]:
        """Clean and tokenize text"""
        if not text:
            return []
        
        # Lowercase
        text = text.lower()
        
        # Normalize abbreviations
        text = re.sub(r'\bpt\b', 'patient', text)
        text = re.sub(r'\bhx\b', 'history', text)
        text = re.sub(r'\bdx\b', 'diagnosis', text)
        
        # Remove special chars
        text = re.sub(r'[^a-z0-9\s\-]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Tokenize
        tokens = word_tokenize(text)
        
        # Filter and lemmatize
        processed = []
        for token in tokens:
            if len(token) < 2:
                continue
            if token in self.stop_words:
                continue
            if token.isdigit() and len(token) > 4:
                continue
            lemma = self.lemmatizer.lemmatize(token)
            processed.append(lemma)
        
        return processed
    
    def encode_tokens(self, tokens: List[str]) -> torch.Tensor:
        """Convert tokens to tensor"""
        indices = [self.vocab.word2idx.get(t, 1) for t in tokens]
        
        # Pad/truncate
        if len(indices) > self.max_seq_length:
            indices = indices[:self.max_seq_length]
        else:
            indices = indices + [0] * (self.max_seq_length - len(indices))
        
        return torch.LongTensor([indices]).to(self.device)
    
    def predict(self, text: str, threshold: float = 0.3, top_k: int = 10) -> List[Dict]:
        """
        Predict ICD-10 codes for input text.
        
        Args:
            text: Medical document text
            threshold: Minimum confidence score
            top_k: Maximum number of predictions
            
        Returns:
            List of predictions with code, confidence, and chapter
        """
        # Preprocess
        tokens = self.preprocess_text(text)
        if len(tokens) == 0:
            return []
        
        # Encode
        X = self.encode_tokens(tokens)
        
        # Predict
        with torch.no_grad():
            probs = self.model(X).cpu().numpy()[0]
        
        # Get predictions above threshold
        predictions = []
        sorted_indices = np.argsort(probs)[::-1]
        
        for idx in sorted_indices:
            if probs[idx] < threshold:
                break
            if len(predictions) >= top_k:
                break
            
            code = self.idx2code[idx]
            chapter = self.icd10_chapters.get(code[0], 'Unknown')
            
            predictions.append({
                'code': code,
                'confidence': float(probs[idx]),
                'chapter': chapter
            })
        
        return predictions
    
    def predict_from_pdf(self, pdf_path: str, threshold: float = 0.3, top_k: int = 10) -> Dict:
        """Extract text from PDF and predict"""
        # Try native extraction first
        text = ""
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text() or ""
                    text += page_text + "\n"
        except Exception as e:
            print(f"PDF extraction error: {e}")
        
        # Fallback to OCR if needed
        if len(text.strip()) < 100:
            try:
                images = convert_from_path(pdf_path, dpi=200)
                text = ""
                for img in images:
                    text += pytesseract.image_to_string(img) + "\n"
            except Exception as e:
                print(f"OCR error: {e}")
        
        predictions = self.predict(text, threshold, top_k)
        
        return {
            'file': os.path.basename(pdf_path),
            'text_length': len(text),
            'predictions': predictions
        }


# Create predictor
predictor = ICD10Predictor(model, vocab, idx2code, device)
print("\n✓ ICD10Predictor ready!")

---

## 🎯 Section 3: Test Predictions (~1 min)

In [ ]:
# Test with sample medical text
sample_text = """
DISCHARGE SUMMARY

Patient presents with bilateral knee pain, worse on the right side. 
History of osteoarthritis of both knees, primary type. 
Patient also has Type 2 diabetes mellitus, well controlled on metformin.
Essential hypertension managed with lisinopril.
Chronic kidney disease stage 2.
Assessment: Primary osteoarthritis bilateral knees, Type 2 DM, HTN, CKD.
Plan: Physical therapy referral, continue current medications.
"""

print("="*60)
print("🧪 TESTING PREDICTION")
print("="*60)
print(f"\nInput text ({len(sample_text)} chars):")
print(sample_text[:200] + "...")

# Predict
predictions = predictor.predict(sample_text, threshold=THRESHOLD, top_k=TOP_K)

print(f"\n📋 Predicted ICD-10 Codes ({len(predictions)}):")
print("-" * 50)
for i, pred in enumerate(predictions, 1):
    print(f"  {i}. {pred['code']:10} | {pred['confidence']:.1%} | {pred['chapter']}")

print("\n" + "="*60)

In [ ]:
# Test with another example
sample_text_2 = """
Patient with history of gout, currently experiencing acute flare in left big toe.
Also noted essential hypertension and hyperlipidemia.
Chronic low back pain with radiculopathy.
Obesity noted, BMI 35.
Recommend colchicine for gout flare, continue statin therapy.
"""

print("\n🧪 TEST CASE 2")
print("="*60)
predictions = predictor.predict(sample_text_2, threshold=0.25)

print(f"📋 Predictions:")
for i, pred in enumerate(predictions, 1):
    print(f"  {i}. {pred['code']:10} | {pred['confidence']:.1%} | {pred['chapter']}")

---

## 📄 Section 4: Predict from PDF Files

In [ ]:
# Predict from a PDF file
# Update this path to your PDF file
PDF_PATH = "/content/drive/MyDrive/485/Other Than 485/"  # Update with actual path

# List available PDFs
pdf_files = [f for f in os.listdir(PDF_PATH) if f.endswith('.pdf')][:5]
print(f"Found {len(pdf_files)} PDFs in folder")
for f in pdf_files:
    print(f"  - {f}")

In [ ]:
# Predict on first PDF
if pdf_files:
    test_pdf = os.path.join(PDF_PATH, pdf_files[0])
    print(f"\n📄 Predicting on: {pdf_files[0]}")
    print("="*60)
    
    result = predictor.predict_from_pdf(test_pdf, threshold=0.25)
    
    print(f"\nText extracted: {result['text_length']} characters")
    print(f"\n📋 Predicted ICD-10 Codes:")
    print("-" * 50)
    for i, pred in enumerate(result['predictions'], 1):
        print(f"  {i}. {pred['code']:10} | {pred['confidence']:.1%} | {pred['chapter']}")
else:
    print("No PDF files found. Update PDF_PATH above.")

---

## 🎮 Section 5: Interactive Demo

In [ ]:
def interactive_predict():
    """
    Interactive prediction demo.
    Enter medical text and get ICD-10 predictions.
    """
    print("\n" + "="*60)
    print("🎮 INTERACTIVE ICD-10 PREDICTOR")
    print("="*60)
    print("Enter medical text below (or 'quit' to exit):")
    print("-" * 60)
    
    while True:
        text = input("\n📝 Enter text: ")
        
        if text.lower() in ['quit', 'exit', 'q']:
            print("\nGoodbye! 👋")
            break
        
        if len(text) < 10:
            print("⚠️ Please enter more text (at least 10 characters)")
            continue
        
        predictions = predictor.predict(text, threshold=0.25, top_k=10)
        
        if not predictions:
            print("\n❌ No predictions above threshold. Try more detailed text.")
        else:
            print(f"\n📋 Predicted ICD-10 Codes:")
            for i, pred in enumerate(predictions, 1):
                print(f"  {i}. {pred['code']:10} | {pred['confidence']:.1%} | {pred['chapter']}")

# Uncomment to run interactive demo:
# interactive_predict()

In [ ]:
# Batch prediction function
def batch_predict(texts: List[str], threshold: float = 0.3) -> List[Dict]:
    """
    Predict ICD-10 codes for multiple texts.
    
    Args:
        texts: List of medical documents
        threshold: Minimum confidence
        
    Returns:
        List of prediction results
    """
    results = []
    for i, text in enumerate(texts):
        preds = predictor.predict(text, threshold=threshold)
        results.append({
            'doc_id': i,
            'text_preview': text[:100] + '...' if len(text) > 100 else text,
            'predictions': preds
        })
    return results

print("✓ Batch prediction function ready!")

---

## 📊 Section 6: Export Functions

In [ ]:
# Export predictions to JSON
def export_predictions(predictions: List[Dict], output_path: str):
    """Save predictions to JSON file"""
    with open(output_path, 'w') as f:
        json.dump(predictions, f, indent=2)
    print(f"✓ Predictions saved to: {output_path}")

# Export predictions to CSV
def export_to_csv(predictions: List[Dict], output_path: str):
    """Save predictions to CSV file"""
    import csv
    
    with open(output_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['doc_id', 'code', 'confidence', 'chapter'])
        
        for result in predictions:
            doc_id = result.get('doc_id', result.get('file', 'unknown'))
            for pred in result['predictions']:
                writer.writerow([
                    doc_id,
                    pred['code'],
                    f"{pred['confidence']:.4f}",
                    pred['chapter']
                ])
    
    print(f"✓ Predictions saved to: {output_path}")

print("✓ Export functions ready!")

---

## ✅ Section 7: Summary

In [ ]:
print("\n" + "="*60)
print("🎉 NOTEBOOK 4 COMPLETE: INFERENCE & DEPLOYMENT")
print("="*60)

print("\n📊 Model Info:")
print(f"  Classes: {config['n_classes']} ICD-10 codes")
print(f"  Best Val F1: {model_info['metrics']['best_val_f1']:.4f}")

print("\n🛠️ Available Functions:")
print("  predictor.predict(text) - Predict from text")
print("  predictor.predict_from_pdf(path) - Predict from PDF")
print("  batch_predict(texts) - Batch prediction")
print("  export_to_csv(results, path) - Export to CSV")

print("\n📌 Usage Example:")
print("  predictions = predictor.predict('Patient has diabetes and hypertension')")
print("  for p in predictions:")
print("      print(p['code'], p['confidence'])")

print("\n" + "="*60)
print("🏁 PROJECT COMPLETE!")
print("="*60)